# Ollama — Quick Introduction

This notebook walks through the basics of using the `ollama` Python library:

1. Instantiating a model
2. Prompting a model
3. Setting up a tool (function calling)
4. Improving performance — disabling thinking mode

In [1]:
# Install the library if needed
# !pip install ollama

---
## 1. Instantiating a Model

Ollama runs models **locally** as a background service (default port `11434`).  
The Python library connects to that service — there is no GPU/model loading in Python itself.

Before running this notebook make sure Ollama is running and the model is pulled:

```bash
ollama pull qwen3:8b
ollama serve          # already running if you installed Ollama normally
```

Creating a "client" is as simple as importing the library.  
You can also point it at a remote Ollama instance by passing a `host`.

In [2]:
import ollama

# Default client — connects to http://localhost:11434
client = ollama.Client()

# To connect to a remote host:
# client = ollama.Client(host="http://192.168.1.10:11434")

# List models that are already downloaded
# NOTE: the library returns typed objects, not plain dicts — use attribute access
models = client.list()
for m in models.models:
    print(m.model)

qwen3-vl-fast:latest
qwen3-vl:8b
llama3.2:latest


We pick one model name and reuse it throughout the notebook.

In [3]:
MODEL = "qwen3-vl:8b"

---
## 2. Prompting a Model

There are two main ways to call a model:

| Method | Description |
|---|---|
| `client.generate(model, prompt)` | Single string in → single string out (no conversation history) |
| `client.chat(model, messages)` | OpenAI-style message list — keeps context across turns |

### 2a. `generate` — the simplest call

In [4]:
response = client.generate(
    model=MODEL,
    prompt="In one sentence, what is a large language model?"
)

print(response.response)

A large language model is a deep learning model trained on vast amounts of text data with billions of parameters, enabling it to understand and generate human-like text across diverse topics.


### 2b. `chat` — multi-turn conversation with a system prompt

Each message is a dict with two keys:
- `role`: `"system"`, `"user"`, or `"assistant"`
- `content`: the text

In [5]:
messages = [
    {"role": "system",  "content": "You are a helpful game development assistant."},
    {"role": "user",    "content": "What is Unity's Update() method used for?"},
]

response = client.chat(model=MODEL, messages=messages)

# The assistant reply
reply = response.message.content
print(reply)

Unity's **`Update()`** method is a core lifecycle function in Unity's `MonoBehaviour` class that is **called once per frame** during the game loop. Here’s what it’s used for and key details to understand:

---

### **Primary Purpose**
- **Frame-rate-dependent logic**: It handles game mechanics that need to respond to **real-time user input or dynamic changes** *each frame*, such as:
  - Player movement (e.g., character control via keyboard/mouse).
  - Input handling (e.g., button presses, mouse clicks).
  - Animation updates (e.g., moving a sprite or playing a looped animation).
  - Camera control or UI interactions (e.g., dragging a UI element).
  - Physics-based effects that aren’t tied to physics calculations (e.g., screen shake).

---

### **How It Works**
- **Called once per frame**: The exact timing depends on your device’s frame rate (e.g., 60 FPS on a high-end PC, lower on mobile). It **does not guarantee consistent timing** (unlike `FixedUpdate()`).
- **Executed after `FixedUp

### 2c. Continuing the conversation

Just append the assistant reply and a new user message to the list and call `chat` again.

In [6]:
# Append the assistant reply to keep history
messages.append({"role": "assistant", "content": reply})

# Follow-up question
messages.append({"role": "user", "content": "And what about FixedUpdate()?"})

response2 = client.chat(model=MODEL, messages=messages)
print(response2.message.content)

Excellent question! **`FixedUpdate()`** is Unity's *critical physics-time method* – and it's **fundamentally different** from `Update()`. Let's break it down clearly:

---

### **What `FixedUpdate()` Does**
- **Called at fixed intervals** (not per frame) to handle **physics calculations**.
- **Timing**:  
  - Default interval = `1/Physics Settings` (e.g., **0.02s** for 60Hz physics updates).  
  - Runs **once per physics timestep**, *not per rendered frame* (e.g., 60 times/sec on 60FPS systems).  
  - **Critical**: It does **NOT depend on frame rate**. Even on a slow device, physics updates happen consistently.

---

### **Why It Exists: Physics vs. Frame Rate**
| **Concept**       | **`Update()`**                          | **`FixedUpdate()`**                      |
|-------------------|----------------------------------------|------------------------------------------|
| **Timing**        | 1 call per rendered frame (variable)   | 1 call per physics timestep (fixed)      |
| **Purpos

---
## 3. Setting Up a Tool (Function Calling)

Ollama supports **tool / function calling**: you describe a Python function to the model and it can decide to call it by returning structured JSON.

The workflow is:
1. Define the Python function.
2. Describe it as a JSON schema (the "tool spec").
3. Pass the tool spec in the `tools` list when calling `chat`.
4. If the model wants to call the tool, the response contains `tool_calls` instead of plain text.
5. Execute the function yourself and send the result back as a `tool` message.
6. Call `chat` again — the model uses the result to give a final answer.

### Step 1 — Define the Python function

In [7]:
def move_object(object_name: str, x: float, y: float, z: float) -> str:
    """
    Simulates moving a Unity GameObject to a new world position.
    In a real project this would send a command to Unity over a socket/HTTP.
    """
    print(f"[TOOL] Moving '{object_name}' to ({x}, {y}, {z})")
    return f"Object '{object_name}' moved to position ({x}, {y}, {z}) successfully."

### Step 2 — Describe the function as a tool spec

In [8]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "move_object",
            "description": "Moves a named GameObject in the Unity scene to the given world-space coordinates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "object_name": {
                        "type": "string",
                        "description": "The name of the Unity GameObject to move."
                    },
                    "x": {"type": "number", "description": "World X position."},
                    "y": {"type": "number", "description": "World Y position."},
                    "z": {"type": "number", "description": "World Z position."}
                },
                "required": ["object_name", "x", "y", "z"]
            }
        }
    }
]

### Steps 3-6 — Full tool-call loop

In [9]:
import json

# Map tool names to actual Python functions
available_tools = {
    "move_object": move_object,
}

messages = [
    {"role": "system", "content": "You control a Unity scene. Use the available tools to fulfill user requests."},
    {"role": "user",   "content": "Please move the Player object to position (3, 0, -5)."},
]

# --- First call: model decides whether to use a tool ---
response = client.chat(model=MODEL, messages=messages, tools=tools)
assistant_message = response.message
print("Model response:", assistant_message)

# Append as a plain dict so the list stays consistent
messages.append({"role": "assistant", "content": assistant_message.content or ""})

# --- Check if the model requested tool calls ---
if assistant_message.tool_calls:
    for tool_call in assistant_message.tool_calls:
        fn_name = tool_call.function.name
        fn_args = tool_call.function.arguments  # already a dict

        print(f"\nCalling tool: {fn_name} with args: {fn_args}")

        # Execute the real Python function
        result = available_tools[fn_name](**fn_args)

        # Send the result back as a 'tool' message
        messages.append({
            "role": "tool",
            "content": result,
        })

    # --- Second call: model generates a final answer using the tool result ---
    final_response = client.chat(model=MODEL, messages=messages)
    print("\nFinal answer:", final_response.message.content)
else:
    # Model answered directly without using a tool
    print("\nDirect answer:", assistant_message.content)

Model response: role='assistant' content='' thinking='Okay, the user wants me to move the Player object to position (3, 0, -5). Let me check the available tools.\n\nThe tool provided is move_object, which requires object_name, x, y, z. The parameters are all required. The user specified the coordinates as (3, 0, -5), so x is 3, y is 0, z is -5. The object name is "Player". \n\nI need to make sure the parameters are correctly mapped. The function expects x, y, z as numbers. The user gave them as a tuple, so 3, 0, -5. \n\nNo other parameters are needed. Just call move_object with object_name "Player" and the coordinates. I think that\'s all. Let me structure the tool call correctly.' images=None tool_calls=[ToolCall(function=Function(name='move_object', arguments={'object_name': 'Player', 'x': 3, 'y': 0, 'z': -5}))]

Calling tool: move_object with args: {'object_name': 'Player', 'x': 3, 'y': 0, 'z': -5}
[TOOL] Moving 'Player' to (3, 0, -5)

Final answer: The Player object has been succes

---
## Summary

| Concept | Key call |
|---|---|
| Connect to Ollama | `ollama.Client()` |
| Simple prompt | `client.generate(model, prompt)` |
| Conversation / system prompt | `client.chat(model, messages)` |
| Tool calling | `client.chat(model, messages, tools=tools)` + handle `tool_calls` in the response |
| Disable thinking mode | `client.chat(..., think=False)` — skip chain-of-thought for faster, simpler tasks |

These primitives are exactly what the Flask server in this project uses to build a multi-agent system on top of Ollama.

In [10]:
import time

msg = [{"role": "user", "content": "Move the Player to (3, 0, -5). Use the move_object tool."}]

# ── With thinking (default) ───────────────────────────────────────────────────
t0 = time.time()
resp_think = client.chat(model=MODEL, messages=msg, tools=tools)          # think=True by default
t_think = time.time() - t0

# ── Without thinking ──────────────────────────────────────────────────────────
t0 = time.time()
resp_no_think = client.chat(model=MODEL, messages=msg, tools=tools, think=False)
t_no_think = time.time() - t0

print(f"With thinking:    {t_think:.1f}s   |  thinking field: {str(resp_think.message.thinking)[:80]}...")
print(f"Without thinking: {t_no_think:.1f}s")
print(f"\nSpeedup: {t_think / t_no_think:.1f}x faster")

# Both should still produce the same tool call
print(f"\nTool called (thinking ON):  {resp_think.message.tool_calls[0].function.name if resp_think.message.tool_calls else 'none'}")
print(f"Tool called (thinking OFF): {resp_no_think.message.tool_calls[0].function.name if resp_no_think.message.tool_calls else 'none'}")

With thinking:    12.7s   |  thinking field: Okay, let me see. The user wants me to move the Player to the coordinates (3, 0,...
Without thinking: 13.6s

Speedup: 0.9x faster

Tool called (thinking ON):  move_object
Tool called (thinking OFF): move_object


### When to keep thinking ON vs OFF

| Situation | Recommendation |
|---|---|
| Tool calling (structured output) | **OFF** — the model just needs to pick a function and fill arguments, no reasoning needed |
| Multi-step agent loops | **OFF** — each step is simple; latency compounds over many rounds |
| Hard math / logic / coding puzzles | **ON** — the chain-of-thought genuinely improves accuracy |
| Creative writing, summaries | **ON or OFF** — negligible accuracy difference; turn off if speed matters |

> **Rule of thumb for agents:** if the task is well-defined and the answer is short (a tool call, a vote, a proposal), always use `think=False`. Reserve thinking for tasks that require deliberation.

In the multi-agent server we built, every agent call uses `think=False`. Because the coordinator runs 5+ rounds of tool calls, and each builder also runs a round, a 5-second thinking overhead per call would make the whole workflow ~10× slower than necessary.

---
## Summary

| Concept | Key call |
|---|---|
| Connect to Ollama | `ollama.Client()` |
| Simple prompt | `client.generate(model, prompt)` |
| Conversation / system prompt | `client.chat(model, messages)` |
| Tool calling | `client.chat(model, messages, tools=tools)` + handle `tool_calls` in the response |

These primitives are exactly what the Flask server in this project uses to build a multi-agent system on top of Ollama.